# Notebook 04 — Statistical Analysis & Inference

**Author:** Section D – Group 8  
**Input:** `data/clean_data.csv`  

**Goal:** Perform rigorous statistical tests to validate market hypotheses and model how physical attributes of a property drive its sale price.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)

df = pd.read_csv('../data/clean_data.csv')
df = df.dropna(subset=['Sale_Price', 'Living_Space_SqFt', 'Bedrooms'])
print(f'Rows for analysis: {len(df):,}')

## 1. Normality Check — Is Price Normally Distributed?
Most parametric tests assume normality. Let's verify using a Q-Q plot and Shapiro-Wilk.

In [ ]:
# Q-Q plot for Sale Price
sample_prices = df['Sale_Price_Capped'].dropna().sample(2000, random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw
stats.probplot(sample_prices, plot=axes[0])
axes[0].set_title('Q-Q Plot: Sale Price (Capped)')

# Log-transformed
stats.probplot(np.log1p(sample_prices), plot=axes[1])
axes[1].set_title('Q-Q Plot: Log(Sale Price)')

plt.tight_layout()
plt.show()

In [ ]:
# Shapiro-Wilk test (on small sample — test is sensitive to N)
stat, p = stats.shapiro(sample_prices.sample(500, random_state=1))
print(f'Shapiro-Wilk: statistic={stat:.4f}, p-value={p:.4g}')
print('→ Rejecting normality. Use non-parametric or log-transformed tests.' if p < 0.05 else '→ Normality cannot be rejected.')

## 2. Hypothesis Test 1 — Does Bedroom Count Affect Price?

**H₀:** Mean sale price of 2-bed homes = mean sale price of 3-bed homes  
**H₁:** They differ significantly  
**Test:** Welch's t-test (does not assume equal variance)

In [ ]:
beds_2 = df[df['Bedrooms'] == 2]['Sale_Price_Capped'].dropna()
beds_3 = df[df['Bedrooms'] == 3]['Sale_Price_Capped'].dropna()

print(f'2-bed properties: {len(beds_2):,} | Mean: ${beds_2.mean():,.0f}')
print(f'3-bed properties: {len(beds_3):,} | Mean: ${beds_3.mean():,.0f}')

In [ ]:
t_stat, p_val = stats.ttest_ind(beds_2, beds_3, equal_var=False)
print(f'Welch T-Statistic : {t_stat:.4f}')
print(f'P-Value           : {p_val:.4g}')
print()
if p_val < 0.05:
    print('✅ Reject H₀: Significant price difference between 2-bed and 3-bed homes (p < 0.05)')
else:
    print('❌ Fail to reject H₀: No significant difference found')

In [ ]:
# Visualize both distributions
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(beds_2, bins=50, alpha=0.6, label='2 Bedrooms', color='steelblue')
ax.hist(beds_3, bins=50, alpha=0.6, label='3 Bedrooms', color='coral')
ax.axvline(beds_2.mean(), color='blue',   linestyle='--', label=f'Mean 2-bed: ${beds_2.mean():,.0f}')
ax.axvline(beds_3.mean(), color='red',    linestyle='--', label=f'Mean 3-bed: ${beds_3.mean():,.0f}')
ax.set_title('Price Distribution: 2-Bed vs 3-Bed Homes', fontsize=13)
ax.set_xlabel('Sale Price ($)')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Hypothesis Test 2 — Single Family Homes vs Condos

**H₀:** No significant price difference between SINGLE_FAMILY and CONDO  
**Test:** Mann-Whitney U (non-parametric, appropriate since prices aren't normal)

In [ ]:
sf_prices    = df[df['property_type'] == 'SINGLE_FAMILY']['Sale_Price_Capped'].dropna()
condo_prices = df[df['property_type'] == 'CONDO']['Sale_Price_Capped'].dropna()

print(f'Single Family: {len(sf_prices):,} listings | Median: ${sf_prices.median():,.0f}')
print(f'Condo        : {len(condo_prices):,} listings | Median: ${condo_prices.median():,.0f}')

In [ ]:
u_stat, p_mw = stats.mannwhitneyu(sf_prices, condo_prices, alternative='two-sided')
print(f'Mann-Whitney U : {u_stat:.4f}')
print(f'P-Value        : {p_mw:.4g}')
if p_mw < 0.05:
    print('✅ Reject H₀: Statistically significant price difference between Single Family and Condo')
else:
    print('❌ Fail to reject H₀')

## 4. ANOVA — Price Across All Property Types

**H₀:** Mean prices are equal across all property types  
**Test:** Kruskal-Wallis (non-parametric one-way ANOVA equivalent)

In [ ]:
groups = [grp['Sale_Price_Capped'].dropna().values for _, grp in df.groupby('property_type')]
h_stat, p_kw = stats.kruskal(*groups)
print(f'Kruskal-Wallis H : {h_stat:.4f}')
print(f'P-Value          : {p_kw:.4g}')
if p_kw < 0.05:
    print('✅ Reject H₀: At least one property type has significantly different prices')
else:
    print('❌ Fail to reject H₀')

## 5. Correlation Analysis

In [ ]:
# Pearson correlation with price
feat_cols = ['Living_Space_SqFt', 'Bedrooms', 'Bathrooms', 'land_space']
print('Pearson Correlation with Sale_Price_Capped:')
for col in feat_cols:
    r, p = stats.pearsonr(df[col].dropna(), df.loc[df[col].notna(), 'Sale_Price_Capped'])
    print(f'  {col:25} r={r:.4f}  p={p:.4g}')

## 6. Simple Linear Regression — Price vs Living Space

In [ ]:
df_reg = df[['Sale_Price_Capped', 'Living_Space_Capped']].dropna()
X = sm.add_constant(df_reg['Living_Space_Capped'])
y = df_reg['Sale_Price_Capped']

model_slr = sm.OLS(y, X).fit()
print(model_slr.summary())

In [ ]:
# Regression line plot
samp = df_reg.sample(2000, random_state=42)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(samp['Living_Space_Capped'], samp['Sale_Price_Capped'],
           alpha=0.3, s=12, color='steelblue', label='Data Points')

x_line = np.linspace(samp['Living_Space_Capped'].min(), samp['Living_Space_Capped'].max(), 100)
y_line = model_slr.params['const'] + model_slr.params['Living_Space_Capped'] * x_line
ax.plot(x_line, y_line, color='red', linewidth=2, label=f'OLS Fit (R²={model_slr.rsquared:.3f})')

ax.set_title('OLS: Sale Price ~ Living Space', fontsize=13)
ax.set_xlabel('Living Space (SqFt)')
ax.set_ylabel('Sale Price ($)')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Multiple Linear Regression — Price Drivers

In [ ]:
features = ['Living_Space_Capped', 'Bedrooms', 'Bathrooms']
df_mlr = df[features + ['Sale_Price_Capped']].dropna()

X_mlr = sm.add_constant(df_mlr[features])
y_mlr = df_mlr['Sale_Price_Capped']

model_mlr = sm.OLS(y_mlr, X_mlr).fit()
print(model_mlr.summary())

In [ ]:
# Model interpretation
coefs = model_mlr.params.drop('const')
print('Price impact per unit increase:')
for name, val in coefs.items():
    direction = '↑' if val > 0 else '↓'
    print(f'  {name:25} {direction} ${val:,.0f}')
print(f'\n  R²: {model_mlr.rsquared:.3f} (model explains {model_mlr.rsquared*100:.1f}% of price variance)')

## 8. Residual Diagnostics

In [ ]:
fitted  = model_mlr.fittedvalues
resids  = model_mlr.resid

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs fitted
axes[0].scatter(fitted, resids, alpha=0.2, s=10, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_title('Residuals vs Fitted Values')
axes[0].set_xlabel('Fitted')
axes[0].set_ylabel('Residuals')

# Residual distribution
axes[1].hist(resids, bins=60, color='coral', edgecolor='white')
axes[1].set_title('Distribution of Residuals')
axes[1].set_xlabel('Residual')

plt.tight_layout()
plt.show()

## 9. Market Segmentation (K-Means Clustering)

Using K-Means to identify distinct property segments based on price and size (e.g., Luxury vs. Budget vs. Entry-Level).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Prepare data for clustering
features_seg = ['Sale_Price_Capped', 'Living_Space_Capped']
X_seg = df[features_seg].dropna().copy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_seg)

# Apply K-Means (k=3)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
X_seg['Segment'] = kmeans.fit_predict(X_scaled)

# Rename segments based on mean price
seg_map = X_seg.groupby('Segment')['Sale_Price_Capped'].mean().sort_values().index
label_map = {seg_map[0]: 'Budget', seg_map[1]: 'Mid-Range', seg_map[2]: 'Luxury'}
X_seg['Segment_Label'] = X_seg['Segment'].map(label_map)

# Visualize segments
plt.figure(figsize=(10, 6))
sns.scatterplot(data=X_seg, x='Living_Space_Capped', y='Sale_Price_Capped', hue='Segment_Label', palette='viridis', alpha=0.6)
plt.title('Market Segmentation: Budget vs Mid-Range vs Luxury', fontsize=13)
plt.xlabel('Living Space (SqFt)')
plt.ylabel('Sale Price ($)')
plt.legend(title='Market Segment')
plt.show()

## 10. Predictive Forecasting (Price Prediction)

In the absence of time-series data, we 'forecast' the market value of a property based on its physical characteristics using a Train/Test split evaluation.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Prepare features and target
feats = ['Living_Space_Capped', 'Bedrooms', 'Bathrooms', 'latitude', 'longitude']
data_model = df[feats + ['Sale_Price_Capped']].dropna()

X_f = data_model[feats]
y_f = data_model['Sale_Price_Capped']

X_train, X_test, y_train, y_test = train_test_split(X_f, y_f, test_size=0.2, random_state=42)

# Train a Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Predictions and Evaluation
preds = rf.predict(X_test)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f'Model Accuracy (R²): {r2:.3f}')
print(f'Mean Absolute Error (Forecast Deviation): ${mae:,.2f}')

# Feature Importance
plt.figure(figsize=(10, 4))
feat_importances = pd.Series(rf.feature_importances_, index=feats).sort_values()
feat_importances.plot(kind='barh', color='teal')
plt.title('Primary Drivers of Price Forecast')
plt.show()